In [ ]:
#!/usr/bin/env python3
"""
DATA255 Lab 1 - NPU Validation Script
Runs MXQ model on validation set and computes accuracy
"""

import argparse
import os
import time
import numpy as np
from PIL import Image
import maccel


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_image(img_path, img_size):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((img_size, img_size), Image.BILINEAR)

    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    arr = np.transpose(arr, (2, 0, 1))
    arr = np.expand_dims(arr, axis=0)

    return arr


def load_validation_list(txt_path):
    items = []
    with open(txt_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            fname, label = line.split()
            items.append((fname, int(label)))
    return items


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mxq", required=True, help="Path to MXQ model")
    parser.add_argument("--val_dir", required=True, help="Validation image folder")
    parser.add_argument("--val_txt", required=True, help="Validation txt file")
    parser.add_argument("--img_size", type=int, default=240)
    args = parser.parse_args()

    if not os.path.exists(args.mxq):
        raise FileNotFoundError("MXQ model not found!")

    model = maccel.Model(args.mxq)

    val_list = load_validation_list(args.val_txt)

    correct = 0
    total = 0

    print("Running NPU inference...")

    start_time = time.time()

    for fname, label in val_list:
        img_path = os.path.join(args.val_dir, fname)
        x = preprocess_image(img_path, args.img_size)

        logits = model.infer(x)

        if isinstance(logits, (list, tuple)):
            logits = logits[0]

        logits = np.array(logits)
        pred = int(np.argmax(logits))

        if pred == label:
            correct += 1

        total += 1

        if total % 100 == 0:
            print(f"Processed {total}/{len(val_list)}")

    end_time = time.time()

    acc = correct / total
    print("\n✅ Validation Accuracy: {:.2f}%".format(acc * 100))
    print("Total Time: {:.2f} sec".format(end_time - start_time))


if __name__ == "__main__":
    main()